# AI Chef Assistant

## Stage 1 Capstone Project

This project combines all the concepts introduced throughout Stage 1 into a single real-world AI application.

### Concepts Covered

- AI Agents
- System Prompts
- Custom Tools
- Web Search
- Short-Term Memory
- Multimodal Inputs
- Image Understanding
- Conversation Threads

The assistant helps users generate personalized recipes based on both text descriptions and uploaded food images while maintaining conversational context.

# Loading Environment Variables

Before building our AI application, we need to securely load our environment variables.

Sensitive information such as API keys should never be hardcoded inside the source code. Instead, they are stored in a `.env` file and loaded at runtime using the `python-dotenv` package.

This approach provides several advantages:

- Keeps API keys secure.
- Prevents accidental exposure when sharing the project.
- Makes switching between environments much easier.
- Follows industry best practices for secret management.

Calling `load_dotenv()` automatically loads all variables defined inside the `.env` file into the current Python environment.

In [37]:
from dotenv import load_dotenv
load_dotenv()

True

# Importing Dependencies

This project combines several components to build a production-like AI application rather than a simple chatbot.

The imported libraries provide the following capabilities:

- **LangChain** for creating the AI agent.
- **LangGraph** for adding short-term memory.
- **SystemMessage** and **HumanMessage** for structured conversations.
- **Tavily Search** to retrieve up-to-date recipe information from the web.
- **FileUpload** to allow users to upload food images directly from the notebook.
- **Base64** encoding for transmitting image data to multimodal models.

Each component plays a specific role, and together they form a complete multimodal AI workflow.

In [38]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from ipywidgets import FileUpload
from IPython.display import display
from langchain.tools import tool
from typing import Any, Dict
from tavily import TavilyClient
import base64

# Building a Custom Web Search Tool

Large Language Models are limited to the knowledge available during training and may not always provide the most recent or specialized information.

To overcome this limitation, we create a custom tool using LangChain's `@tool` decorator.

The `web_search` tool allows the agent to search the web for cooking recipes based on the ingredients provided by the user.

Instead of relying solely on the model's internal knowledge, the agent can dynamically retrieve external information whenever necessary.

This is one of the key ideas behind modern AI Agents:

> The model reasons about *when* to use a tool, while the tool performs the external task.

By integrating web search, the assistant becomes more accurate and capable of suggesting recipes beyond its static knowledge.

In [39]:
travily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Use this tool to search the web for cooking recipes based on a specific list of available ingredients.
     The input should be a highly targeted search query, such as 'easy recipes using chicken, tomatoes, and garlic'.
      Use this tool ONLY when you need to find real-world recipes,
       cooking instructions, or meal ideas for the ingredients provided by the user."""
    response = travily_client.search(query)
    return response['results']

# Defining the Agent's Role

Before creating the agent, we define its behavior using a `SystemMessage`.

The system prompt acts as the agent's permanent instruction set, defining its personality, responsibilities, and decision-making process.

In this project, the assistant is designed to behave as a professional **Master Chef Assistant**.

The prompt clearly specifies:

- The assistant's objective.
- How recipes should be generated.
- When to use the web search tool.
- How to analyze uploaded food images.
- How to respond when ingredients are insufficient.

Providing detailed system instructions significantly improves consistency and enables the agent to make better decisions during conversations.

In [40]:
system_prompt = SystemMessage(content="""You are a highly creative and resourceful Master Chef Assistant. Your primary goal is to help users cook delicious meals using ONLY the ingredients they currently have in their fridge or pantry.

When a user provides a list of ingredients or an image of their fridge, you must follow these steps:
1. Analyze the input to identify all available ingredients. You may assume the user has basic pantry staples (e.g., salt, black pepper, cooking oil, water).
2. Formulate a highly targeted search query based on these ingredients.
3. USE THE SEARCH TOOL to find real, practical, and highly-rated recipes matching the ingredients.
4. Select 1 to 3 distinct recipe options to present to the user.

For each recipe, your response MUST include:
- A catchy and appetizing title.
- A brief explanation of why this recipe is a great fit for their available ingredients.
- Preparation and cooking time (if available).
- A clear, step-by-step guide on how to cook it.

Constraints:
- DO NOT hallucinate recipes; rely on the search tool results.
- DO NOT suggest recipes that require main ingredients the user explicitly does not have.
- Always maintain an encouraging, friendly, and culinary-expert tone.
- Reply to the user in the same language they used to speak to you (e.g., if they speak Arabic, provide the recipes and instructions in Arabic).""")

# Creating the AI Chef Agent

Now we combine all previously defined components into a single AI Agent.

The agent is configured with:

- A language model (`gpt-5-nano`).
- The custom `web_search` tool.
- An `InMemorySaver` checkpointer for short-term memory.
- The custom system prompt.

This architecture enables the assistant to:

- Remember previous interactions.
- Search the web whenever additional information is required.
- Follow the cooking rules defined in the system prompt.

Instead of acting as a standalone language model, the agent becomes an intelligent system capable of reasoning, using tools, and maintaining conversational context.

In [41]:
chef_agent= create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    checkpointer=InMemorySaver(),
    system_prompt= system_prompt
)


# Uploading Food Images

To make the assistant multimodal, users are given the ability to upload an image of the available ingredients.

The uploaded image will later be analyzed together with the user's textual request.

This enables more natural interactions, as users no longer need to manually list every ingredient.

Supporting image inputs is one of the major advantages of modern multimodal AI systems compared to traditional text-only chatbots.

In [42]:
uploader = FileUpload(accept='.JPEG', multiple=False)
display(uploader)

FileUpload(value=(), accept='.JPEG', description='Upload')

# Preparing the Image for the Model

After an image is uploaded, it cannot be sent directly to the language model.

Instead, the image must first be converted into a Base64 encoded string.

The process includes:

1. Reading the uploaded image.
2. Extracting its binary content.
3. Encoding the binary data using Base64.

This encoded representation allows binary media to be safely transmitted inside API requests.

If no image is uploaded, the assistant gracefully falls back to using only the textual description provided by the user.

In [43]:
img_b64 = None

if uploader.value:
    uploaded_file = uploader.value[0]
    
    content_mv = uploaded_file["content"]
    img_bytes = bytes(content_mv)
    
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
    print("تم معالجة الصورة بنجاح!")
else:
    print("لم يتم رفع صورة، سيتم الاعتماد على النص فقط.")

لم يتم رفع صورة، سيتم الاعتماد على النص فقط.


# Constructing a Multimodal User Message

The user's request is represented as a structured message rather than a plain string.

The message may contain multiple modalities:

- Text describing the available ingredients.
- An uploaded image of the ingredients.

Each modality is added to the same message object, allowing the model to reason about both sources of information simultaneously.

This demonstrates one of the most powerful capabilities of modern AI models: combining multiple input types within a single inference process.

In [44]:
message_content = []

user_text = "I have chicken breasts, tomatoes, onions, garlic, and heavy cream. What can I make with these?"

if user_text:
    message_content.append(
        {"type": "text", "text": user_text}
    )

if img_b64:
    message_content.append(
        {
            "type": "image_url", 
            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
        }
    )

multimodal_question = HumanMessage(content=message_content)


# Running the Complete AI Workflow

Finally, we execute the AI Chef Assistant.

A `thread_id` is provided so the agent can preserve conversation history across multiple interactions using LangGraph's short-term memory.

During execution, the agent performs several tasks:

- Understands the user's request.
- Analyzes the uploaded image (if available).
- Decides whether web search is necessary.
- Retrieves additional recipe information when appropriate.
- Generates personalized cooking recommendations.

This project demonstrates how multiple AI concepts—including agents, memory, tools, and multimodal inputs—can be combined into a single real-world application.

In [45]:
config = {"configurable": {"thread_id": "1"}}

response = chef_agent.invoke(
    {"messages": [multimodal_question]}, config=config
)

print(response["messages"][-1].content)

Nice lineup of ingredients! Here are 2 real, tasty options that line up perfectly with what you have (chicken breasts, tomatoes, onions, garlic, heavy cream). Both are quick, comforting, and mostly one-pan.

Option 1: One-Pan Creamy Tomato Chicken
- Why it fits: It’s a quick, beginner-friendly skillet that starts with searing chicken, then builds a creamy tomato sauce right in the same pan. You get rich flavor from garlic, onions, and cream without needing extra ingredients.
- Time: about 30 minutes total (prep ~5; cook ~25)

Step-by-step:
1) Pat the chicken dry and season generously with salt and pepper.
2) Heat a couple of tablespoons of oil in a large skillet over medium-high. Sear the chicken on both sides until golden brown but not fully cooked through. Remove to a plate.
3) In the same pan, add a little more oil if needed. Sauté the chopped onion until translucent, about 3–4 minutes.
4) Add minced garlic and cook 30–60 seconds until fragrant.
5) Add chopped tomatoes (or a quick s